# Text to SQL with Nugen

This cookbook demonstrates how to turn natural language questions into executable SQL queries using Nugen. 
We will replicate a robust pattern that includes:
1.  **Schema Definition**: Giving the model context about your database.
2.  **Chain of Thought (CoT)**: Encouraging the model to plan before writing code.
3.  **Execution & Result**: Running the query against a real SQLite database.


## 1. Setup & Database Initialization
First, we'll install dependencies and set up a sample SQLite database with `employees` and `departments`.

In [ ]:
%pip install requests python-dotenv pandas

import sqlite3
import pandas as pd
import requests
import os
import json
from dotenv import load_dotenv

# Load API Key
load_dotenv()
NUGEN_API_KEY = os.getenv("NUGEN_API_KEY")
BASE_URL = "https://api.nugen.in/inference/chat/completions"
MODEL = "nugen-flash-instruct"

if not NUGEN_API_KEY:
    raise ValueError("Please set NUGEN_API_KEY in your .env file")

# Create in-memory DB
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Create Tables
cursor.executescript("""
    CREATE TABLE departments (
        id INTEGER PRIMARY KEY,
        name TEXT,
        location TEXT
    );
    
    CREATE TABLE employees (
        id INTEGER PRIMARY KEY,
        name TEXT,
        age INTEGER,
        department_id INTEGER,
        salary REAL,
        hire_date DATE,
        FOREIGN KEY (department_id) REFERENCES departments(id)
    );
    
    INSERT INTO departments (id, name, location) VALUES
    (1, 'Engineering', 'San Francisco'),
    (2, 'Sales', 'New York'),
    (3, 'Marketing', 'London');

    INSERT INTO employees (name, age, department_id, salary, hire_date) VALUES
    ('Alice', 30, 1, 120000, '2022-01-15'),
    ('Bob', 35, 2, 95000, '2021-05-20'),
    ('Charlie', 28, 1, 115000, '2023-03-10'),
    ('David', 42, 3, 80000, '2020-11-05'),
    ('Eve', 31, 2, 98000, '2022-08-14');
""")
conn.commit()
print("Database initialized.")

## 2. Define the Schema Helper
The model works best when provided with a clear schema in a format it can understand (like XML or simple text).

In [ ]:
def get_database_schema(conn):
    """Extracts schema info from SQLite and formats it as an XML string."""
    schema_str = "<schema>\n"
    cursor = conn.cursor()
    
    # Get tables
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [row[0] for row in cursor.fetchall()]
    
    for table in tables:
        cursor.execute(f"PRAGMA table_info({table})")
        columns = cursor.fetchall()
        col_strs = [f"{col[1]} ({col[2]})" for col in columns]
        schema_str += f"  Table: {table}\n  Columns: {', '.join(col_strs)}\n\n"
        
    schema_str += "</schema>"
    return schema_str

print(get_database_schema(conn))

## 3. The Prompting Function
We'll create a function `text_to_sql` that constructs a prompt using the schema and the user's question.
We use **<thought_process>** tags to encourage the model to reason before writing SQL, which improves accuracy for complex joins.

In [ ]:
def generate_sql(question, schema_context):
    system_prompt = """You are an expert SQL assistant. 
Your goal is to output valid SQLite queries based on the provided schema.
1. Always wrap your reasoning in <thought_process> tags.
2. Always wrap the final SQL query in <sql> tags.
3. Only use tables and columns defined in the schema.
4. Do not include markdown formatting (```sql) outside the tags.
"""
    
    user_prompt = f"""
Here is the database schema:
{schema_context}

Question: {question}

Please provide the <thought_process> followed by the <sql> query.
"""
    
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.1
    }
    
    try:
        response = requests.post(BASE_URL, json=payload, headers={
            "Authorization": f"Bearer {NUGEN_API_KEY}",
            "Content-Type": "application/json"
        })
        response.raise_for_status()
        return response.json()['choices'][0]['message']['content']
    except Exception as e:
        print(f"API Error: {e}")
        return None

## 4. Execution Loop
Here we parse the output to extract the SQL inside the `<sql>` tags and execute it.

In [ ]:
import re

def extract_and_run(llm_response):
    # Extract thought process
    thoughts = re.search(r'<thought_process>(.*?)</thought_process>', llm_response, re.DOTALL)
    if thoughts:
        print(f"🤔 Reasoning:\n{thoughts.group(1).strip()}\n")
    
    # Extract SQL
    sql_match = re.search(r'<sql>(.*?)</sql>', llm_response, re.DOTALL)
    if not sql_match:
        print("❌ No SQL found in response.")
        return
        
    sql_query = sql_match.group(1).strip()
    print(f"💻 Executing SQL:\n{sql_query}\n")
    
    try:
        results = pd.read_sql_query(sql_query, conn)
        print("📊 Results:")
        display(results)
    except Exception as e:
        print(f"❌ SQL Execution Error: {e}")

## 5. Examples
Let's test with a few queries ranging from simple to complex.

In [ ]:
# Example 1: Simple Select
schema = get_database_schema(conn)
response = generate_sql("Show me the names of all employees in the Engineering department.", schema)
extract_and_run(response)

In [ ]:
# Example 2: Aggregation
response = generate_sql("What is the average salary by department?", schema)
extract_and_run(response)

In [ ]:
# Example 3: Complex Filter
response = generate_sql("List employees hired after 2021 who earn more than 100k.", schema)
extract_and_run(response)